# Batch Parameter Management

This notebook demonstrates how to read and modify HMS basin parameters in bulk
using DataFrames. Instead of looping element-by-element, you can:

1. **Export** all parameters to a DataFrame or CSV
2. **Edit** in Excel, pandas, or any tabular tool
3. **Import** changes back with automatic backup and validation

| Method | Purpose |
|--------|---------|
| `get_all_loss_parameters()` | Read loss params for ALL subbasins |
| `get_all_transform_parameters()` | Read transform params for ALL subbasins |
| `get_all_baseflow_parameters()` | Read baseflow params for ALL subbasins |
| `get_all_routing_parameters()` | Read routing params for ALL reaches |
| `set_all_*_parameters()` | Write modified DataFrame back to file |
| `export_parameters_csv()` | Export to CSV for Excel editing |
| `import_parameters_csv()` | Import CSV back to basin file |
| `HmsMet.set_all_gage_assignments()` | Batch update gage assignments |

**Estimated Time**: 10 minutes

In [1]:
# pip install hms-commander

**For Development**: If working on hms-commander source code, use the `hmscmdr_local`
conda environment (editable install) instead of pip install.

In [2]:
import shutil
from pathlib import Path
from hms_commander import HmsExamples, HmsBasin, HmsMet

print("hms-commander loaded")

hms-commander loaded


## 1. Extract Example Project

We use the **tenk** (Tenkiller Lake) project which has 4 subbasins with
Initial+Constant loss, Modified Clark transform, Recession baseflow,
and 5 reaches with Modified Puls and Lag routing.

In [3]:
# Extract a fresh copy for this notebook
project_path = HmsExamples.extract_project(
    "tenk",
    output_path=Path.cwd() / 'hms_example_projects' / 'tenk_batch'
)

# Find the basin file
basin_file = next(Path(project_path).glob('*.basin'))
print(f"Basin file: {basin_file.name}")

2026-02-21 11:22:18 - hms_commander.HmsExamples - INFO - Found HMS 4.10 at C:\Program Files\HEC\HEC-HMS\4.10


2026-02-21 11:22:18 - hms_commander.HmsExamples - INFO - Found HMS 4.11 at C:\Program Files\HEC\HEC-HMS\4.11


2026-02-21 11:22:18 - hms_commander.HmsExamples - INFO - Found HMS 4.12 at C:\Program Files\HEC\HEC-HMS\4.12


2026-02-21 11:22:18 - hms_commander.HmsExamples - INFO - Found HMS 4.13 at C:\Program Files\HEC\HEC-HMS\4.13


2026-02-21 11:22:18 - hms_commander.HmsExamples - INFO - Found HMS 4.4.1 at C:\Program Files\HEC\HEC-HMS\4.4.1


...

2026-02-21 11:22:18 - hms_commander.HmsExamples - INFO - Extracting 'tenk' from HMS 4.13


2026-02-21 11:22:18 - hms_commander.HmsExamples - INFO - Source: C:\Program Files\HEC\HEC-HMS\4.13\samples.zip


2026-02-21 11:22:18 - hms_commander.HmsExamples - INFO - Destination: C:\GH\hms-commander\examples\hms_example_projects\tenk_batch\tenk


2026-02-21 11:22:18 - hms_commander.HmsExamples - INFO - Successfully extracted 'tenk' to C:\GH\hms-commander\examples\hms_example_projects\tenk_batch\tenk


Basin file: Tenk_1.basin


## 2. Read All Loss Parameters

Instead of calling `get_loss_parameters()` for each subbasin individually,
`get_all_loss_parameters()` returns a single DataFrame with one row per subbasin.
Parameters not applicable to a subbasin's loss method appear as NaN.

In [4]:
loss_df = HmsBasin.get_all_loss_parameters(basin_file)

print(f"Shape: {loss_df.shape[0]} subbasins x {loss_df.shape[1]} columns")
print(f"Loss methods: {loss_df['loss_method'].unique()}")
print()

# Show the relevant columns (drop all-NaN columns for display)
display_df = loss_df.dropna(axis=1, how='all')
display_df

2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Read 4 Subbasin parameter records from Tenk_1.basin


Shape: 4 subbasins x 22 columns
Loss methods: ['Initial+Constant']



,name,area,loss_method,percent_impervious,initial_loss
0,86,635.0,Initial+Constant,0.0,1.00
1,85,324.0,Initial+Constant,0.0,1.00
2,113,307.0,Initial+Constant,0.0,1.30
3,127,345.0,Initial+Constant,10.0,1.15


## 3. Read All Transform Parameters

In [5]:
transform_df = HmsBasin.get_all_transform_parameters(basin_file)

print(f"Transform methods: {transform_df['transform_method'].unique()}")
transform_df.dropna(axis=1, how='all')

2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Read 4 Subbasin parameter records from Tenk_1.basin


Transform methods: ['Modified Clark']


,name,area,transform_method,time_of_concentration,storage_coefficient
0,86,635.0,Modified Clark,24.0,11.6
1,85,324.0,Modified Clark,30.0,15.5
2,113,307.0,Modified Clark,18.0,8.7
3,127,345.0,Modified Clark,1.0,7.0


## 4. Read All Routing Parameters

In [6]:
routing_df = HmsBasin.get_all_routing_parameters(basin_file)

print(f"Routing methods: {routing_df['route_method'].unique()}")
routing_df.dropna(axis=1, how='all')

2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Read 5 Reach parameter records from Tenk_1.basin


Routing methods: ['Modified Puls' 'Lag']


,name,route_method,number_of_reaches,storage_outflow_table_name,initial_variable,channel_loss,lag
0,R-1,Modified Puls,6.0,Table 4,Combined Inflow,None,NaN
1,R-2,Modified Puls,20.0,Table 3,Combined Inflow,None,NaN
2,R-3,Modified Puls,4.0,Table 1,Combined Inflow,None,NaN
3,R-4,Modified Puls,4.0,Table 2,Combined Inflow,None,NaN
4,R-5,Lag,NaN,NaN,Combined Inflow,None,120.0


## 5. Read All Baseflow Parameters

In [7]:
baseflow_df = HmsBasin.get_all_baseflow_parameters(basin_file)

print(f"Baseflow methods: {baseflow_df['baseflow_method'].unique()}")
baseflow_df.dropna(axis=1, how='all')

2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Read 4 Subbasin parameter records from Tenk_1.basin


Baseflow methods: ['Recession']


,name,area,baseflow_method,recession_factor
0,86,635.0,Recession,0.79
1,85,324.0,Recession,0.79
2,113,307.0,Recession,0.79
3,127,345.0,Recession,0.79


## 6. Modify Parameters in Bulk

Edit the DataFrame, then write it back. Only non-NaN values that differ
from the file are written. A `.bak` backup is created automatically.

### Example: Increase initial loss by 20% for all subbasins

In [8]:
# Read current values
loss_df = HmsBasin.get_all_loss_parameters(basin_file)

print("Before:")
print(loss_df[['name', 'initial_loss']].to_string(index=False))

# Modify: increase initial_loss by 20%
loss_df['initial_loss'] = loss_df['initial_loss'] * 1.2

print("\nAfter (DataFrame):")
print(loss_df[['name', 'initial_loss']].to_string(index=False))

2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Read 4 Subbasin parameter records from Tenk_1.basin


Before:
name  initial_loss
  86          1.00
  85          1.00
 113          1.30
 127          1.15

After (DataFrame):
name  initial_loss
  86          1.20
  85          1.20
 113          1.56
 127          1.38


In [9]:
# Write changes back to the basin file
result = HmsBasin.set_all_loss_parameters(basin_file, loss_df, create_backup=True)

print(f"Elements modified: {result['elements_modified']}")
print(f"Parameters changed: {result['parameters_changed']}")
print(f"Backup created: {result['backup_path']}")

if result['elements_not_found']:
    print(f"Not found: {result['elements_not_found']}")
if result['warnings']:
    print(f"Warnings: {result['warnings']}")

2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Created backup: C:\GH\hms-commander\examples\hms_example_projects\tenk_batch\tenk\Tenk_1.basin.bak


2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Updated 4 Subbasins, 4 parameters changed


Elements modified: 4
Parameters changed: 4
Backup created: C:\GH\hms-commander\examples\hms_example_projects\tenk_batch\tenk\Tenk_1.basin.bak


In [10]:
# Verify: read back and confirm
verify_df = HmsBasin.get_all_loss_parameters(basin_file)

print("Verified from file:")
print(verify_df[['name', 'initial_loss']].to_string(index=False))

2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Read 4 Subbasin parameter records from Tenk_1.basin


Verified from file:
name  initial_loss
  86          1.20
  85          1.20
 113          1.56
 127          1.38


### Idempotent Writes

Setting the same values again produces zero changes — the setter compares
numerically, so `"1"` (integer in file) equals `1.0` (float in DataFrame).

In [11]:
# Write the same values again — should report 0 changes
result = HmsBasin.set_all_loss_parameters(basin_file, verify_df, create_backup=False)

print(f"Parameters changed: {result['parameters_changed']}  (expected: 0)")

2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Updated 0 Subbasins, 0 parameters changed


Parameters changed: 0  (expected: 0)


### Partial Updates (NaN = Skip)

You can update a single parameter for a single subbasin. Columns not in the
DataFrame (or with NaN values) are left unchanged.

In [12]:
import pandas as pd

# Update only percent_impervious for subbasin 127
partial_df = pd.DataFrame({
    'name': ['127'],
    'percent_impervious': [15.0]  # was 10.0
})

result = HmsBasin.set_all_loss_parameters(basin_file, partial_df, create_backup=False)
print(f"Changed: {result['parameters_changed']} parameter(s) in {result['elements_modified']} element(s)")

# Verify only that value changed
check = HmsBasin.get_all_loss_parameters(basin_file)
print(check[['name', 'percent_impervious', 'initial_loss']].to_string(index=False))

2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Updated 1 Subbasins, 1 parameters changed


2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Read 4 Subbasin parameter records from Tenk_1.basin


Changed: 1 parameter(s) in 1 element(s)
name  percent_impervious  initial_loss
  86                 0.0          1.20
  85                 0.0          1.20
 113                 0.0          1.56
 127                15.0          1.38


## 7. CSV Roundtrip: Export → Edit in Excel → Import

The most common batch workflow: export all parameters to CSV, edit in Excel
or a text editor, then import the changes back.

### Export

In [13]:
# Reset to fresh copy first
shutil.rmtree(project_path, ignore_errors=True)
project_path = HmsExamples.extract_project(
    "tenk",
    output_path=Path.cwd() / 'hms_example_projects' / 'tenk_batch'
)
basin_file = next(Path(project_path).glob('*.basin'))

# Export all parameter types to CSV
csv_dir = Path.cwd() / 'hms_example_projects' / 'tenk_batch_csv'
csv_dir.mkdir(exist_ok=True)
csv_path = csv_dir / 'tenk_params.csv'

HmsBasin.export_parameters_csv(basin_file, csv_path)

# Show what files were created
print("Exported CSV files:")
for f in sorted(csv_dir.glob('tenk_params*.csv')):
    print(f"  {f.name} ({f.stat().st_size:,} bytes)")

2026-02-21 11:22:18 - hms_commander.HmsExamples - INFO - Using latest installed version: 4.13


2026-02-21 11:22:18 - hms_commander.HmsExamples - INFO - Extracting 'tenk' from HMS 4.13


2026-02-21 11:22:18 - hms_commander.HmsExamples - INFO - Source: C:\Program Files\HEC\HEC-HMS\4.13\samples.zip


2026-02-21 11:22:18 - hms_commander.HmsExamples - INFO - Destination: C:\GH\hms-commander\examples\hms_example_projects\tenk_batch\tenk


2026-02-21 11:22:18 - hms_commander.HmsExamples - INFO - Successfully extracted 'tenk' to C:\GH\hms-commander\examples\hms_example_projects\tenk_batch\tenk


...

2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Exported transform parameters to C:\GH\hms-commander\examples\hms_example_projects\tenk_batch_csv\tenk_params_transform.csv


2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Exported baseflow parameters to C:\GH\hms-commander\examples\hms_example_projects\tenk_batch_csv\tenk_params_baseflow.csv


2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Exported routing parameters to C:\GH\hms-commander\examples\hms_example_projects\tenk_batch_csv\tenk_params_routing.csv


C:\GH\hms-commander\hms_commander\HmsBasin.py:1164: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(combined_parts, ignore_index=True, sort=False)
2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Exported parameters to C:\GH\hms-commander\examples\hms_example_projects\tenk_batch_csv\tenk_params.csv


Exported CSV files:
  tenk_params.csv (3,376 bytes)
  tenk_params_baseflow.csv (830 bytes)
  tenk_params_loss.csv (763 bytes)
  tenk_params_routing.csv (953 bytes)
  tenk_params_transform.csv (543 bytes)


In [14]:
# Peek at the loss CSV
loss_csv = csv_dir / 'tenk_params_loss.csv'
print(f"Contents of {loss_csv.name}:")
print(loss_csv.read_text()[:500])

Contents of tenk_params_loss.csv:
# HMS Basin Parameters - loss
# Source: Tenk_1.basin
# Exported: 2026-02-21 11:22:18
# Import with: HmsBasin.import_parameters_csv("Tenk_1.basin", "tenk_params_loss.csv")
param_type,name,area,loss_method,percent_impervious,initial_loss,moisture_deficit,wetting_front_suction,hydraulic_conductivity,initial_variable,initial_deficit,maximum_deficit,constant_rate,percolation_rate,curve_number,initial_abstraction,initial_rate,saturated_conductivity,wetting_front_capillary,soil,groundwater_1,groundwate


### Import (No Modifications = Zero Changes)

In [15]:
# Import the unmodified CSV — should produce 0 changes
result = HmsBasin.import_parameters_csv(basin_file, csv_path, create_backup=True)

for ptype, summary in result.items():
    print(f"{ptype}: {summary['elements_modified']} modified, "
          f"{summary['parameters_changed']} params changed")

2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Created backup: C:\GH\hms-commander\examples\hms_example_projects\tenk_batch\tenk\Tenk_1.basin.bak


2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Updated 0 Subbasins, 0 parameters changed


2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Updated 0 Subbasins, 0 parameters changed


2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Updated 0 Reachs, 0 parameters changed


2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Updated 0 Subbasins, 0 parameters changed


2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Imported parameters from C:\GH\hms-commander\examples\hms_example_projects\tenk_batch_csv\tenk_params.csv


baseflow: 0 modified, 0 params changed
loss: 0 modified, 0 params changed
routing: 0 modified, 0 params changed
transform: 0 modified, 0 params changed


### Import After Editing

Simulate what happens after a user edits the CSV in Excel.

In [16]:
import pandas as pd

# Read the loss CSV, modify a value, and write it back
edited_df = pd.read_csv(loss_csv, comment='#')

# Simulate an engineer changing initial_loss for subbasin 113
edited_df.loc[edited_df['name'].astype(str) == '113', 'initial_loss'] = 2.0

# Write modified CSV (preserving comment headers)
header = [line for line in loss_csv.read_text().split('\n') if line.startswith('#')]
csv_text = edited_df.to_csv(index=False)
loss_csv.write_text('\n'.join(header) + '\n' + csv_text)

print("Modified CSV (initial_loss for 113 changed to 2.0):")
print(edited_df[['name', 'initial_loss']].to_string(index=False))

Modified CSV (initial_loss for 113 changed to 2.0):
 name  initial_loss
   86          1.00
   85          1.00
  113          2.00
  127          1.15


In [17]:
# Import the modified loss CSV
result = HmsBasin.import_parameters_csv(basin_file, loss_csv, create_backup=False)

for ptype, summary in result.items():
    print(f"{ptype}: {summary['elements_modified']} modified, "
          f"{summary['parameters_changed']} params changed")

# Verify
check = HmsBasin.get_all_loss_parameters(basin_file)
print(f"\nVerified initial_loss for 113: {check.loc[check['name'] == '113', 'initial_loss'].values[0]}")

2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Updated 0 Subbasins, 0 parameters changed


2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Imported parameters from C:\GH\hms-commander\examples\hms_example_projects\tenk_batch_csv\tenk_params_loss.csv


2026-02-21 11:22:18 - hms_commander.HmsBasin - INFO - Read 4 Subbasin parameter records from Tenk_1.basin


loss: 0 modified, 0 params changed



Verified initial_loss for 113: 1.3


## 8. Batch Met Operations: Gage Assignments

`HmsMet.set_all_gage_assignments()` is the batch write counterpart to
`get_gage_assignments()`. It updates precipitation gage assignments for
multiple subbasins in a single call.

In [18]:
# Find the met file
met_files = list(Path(project_path).glob('*.met'))
if met_files:
    met_file = met_files[0]
    print(f"Met file: {met_file.name}")
    
    # Read current assignments
    assignments = HmsMet.get_gage_assignments(met_file)
    print(f"\nCurrent gage assignments ({len(assignments)} subbasins):")
    display(assignments)
else:
    print("No met file found in project")

2026-02-21 11:22:18 - hms_commander.HmsMet - INFO - Reading gage assignments from: C:\GH\hms-commander\examples\hms_example_projects\tenk_batch\tenk\Stage3_HRAP.met


2026-02-21 11:22:18 - hms_commander.HmsMet - INFO - Found 4 gage assignments


Met file: Stage3_HRAP.met

Current gage assignments (4 subbasins):


,subbasin,precip_gage,weight
0,113,None,1.0
1,127,None,1.0
2,85,None,1.0
3,86,None,1.0


In [19]:
if met_files and not assignments.empty:
    # Write back unchanged — should report 0 modifications
    result = HmsMet.set_all_gage_assignments(met_file, assignments, create_backup=False)
    print(f"Subbasins modified: {result['subbasins_modified']}  (expected: 0)")
    
    if result['subbasins_not_found']:
        print(f"Not found: {result['subbasins_not_found']}")

2026-02-21 11:22:18 - hms_commander.HmsMet - INFO - Updated 0 gage assignments in Stage3_HRAP.met


Subbasins modified: 0  (expected: 0)


## 9. Cleanup

In [20]:
# Remove extracted example projects
shutil.rmtree(Path.cwd() / 'hms_example_projects' / 'tenk_batch', ignore_errors=True)
shutil.rmtree(Path.cwd() / 'hms_example_projects' / 'tenk_batch_csv', ignore_errors=True)
print("Cleaned up temporary files")

Cleaned up temporary files


## Summary

| Operation | Method | Input | Output |
|-----------|--------|-------|--------|
| Read all loss params | `get_all_loss_parameters()` | basin path | DataFrame |
| Read all transform params | `get_all_transform_parameters()` | basin path | DataFrame |
| Read all baseflow params | `get_all_baseflow_parameters()` | basin path | DataFrame |
| Read all routing params | `get_all_routing_parameters()` | basin path | DataFrame |
| Write params from DataFrame | `set_all_*_parameters()` | basin path + DataFrame | summary dict |
| Export to CSV | `export_parameters_csv()` | basin path + CSV path | CSV files |
| Import from CSV | `import_parameters_csv()` | basin path + CSV path | summary dict |
| Batch gage assignments | `HmsMet.set_all_gage_assignments()` | met path + DataFrame | summary dict |

**Key Design Decisions**:
- **NaN = skip**: Only non-NaN values are written, making partial updates safe
- **Idempotent**: Setting the same values reports 0 changes (numeric comparison)
- **Auto-backup**: `.bak` file created before any write (`create_backup=True` default)
- **Summary dict**: Every write returns counts of what changed

## Next Steps

- **03_file_ops_basin_met_control_gage.ipynb**: Single-element file operations
- **05_clone_workflow.ipynb**: Non-destructive cloning for QAQC
- **15_upstream_network_analysis.ipynb**: Network traversal and drainage areas